In [ ]:
# 1. INSTALL DEPENDENCIES
!pip install tree-sitter==0.21.3 tree-sitter-languages==1.10.2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install tree-sitter==0.21.3 tree-sitter-languages==1.10.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 114.0 MB/s eta 0:00:00


In [ ]:
import os
import gc
import json
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import tree_sitter_languages
from datasets import Dataset
from transformers import (
    RobertaTokenizer, RobertaModel, RobertaPreTrainedModel,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from google.colab import drive

# --- 1. CONFIGURATION ---
drive.mount('/content/drive')
MODEL_NAME = "microsoft/unixcoder-base"
BASE_DIR = "/content/drive/MyDrive/Task C"
MAX_LENGTH = 512
BATCH_SIZE = 128
LEARNING_RATE = 3e-5
LOGGING_STEPS = 1000
EVAL_STEPS = 1000

os.makedirs(BASE_DIR, exist_ok=True)

# --- 2. ADVANCED CANONICALIZER (From your original code) ---
class UniversalCanonicalizer:
    def __init__(self):
        self.lang_map = {
            'python': 'python', 'py': 'python', 'java': 'java', 'cpp': 'cpp',
            'c++': 'cpp', 'c': 'c', 'c#': 'c_sharp', 'cs': 'c_sharp',
            'javascript': 'javascript', 'js': 'javascript', 'php': 'php', 'go': 'go'
        }
        self.parsers = {}

    def _get_parser(self, lang_key):
        parser_name = self.lang_map.get(lang_key, 'python')
        if parser_name in self.parsers: return self.parsers[parser_name]
        try:
            parser = tree_sitter_languages.get_parser(parser_name)
            self.parsers[parser_name] = parser
            return parser
        except: return None

    def canonicalize(self, code_str, lang_label):
        if not isinstance(code_str, str) or code_str == "": return ""
        lang_key = str(lang_label).lower()
        parser = self._get_parser(lang_key)
        if not parser: return code_str

        code_bytes = code_str.encode('utf8')
        try:
            tree = parser.parse(code_bytes)
            code_bytes = self._rename_identifiers(code_bytes, tree.root_node)
            canon_code = code_bytes.decode('utf8', errors='ignore')
            return self._flatten_layout(canon_code, self.lang_map.get(lang_key, 'python'))
        except: return code_str

    def _rename_identifiers(self, code_bytes, root_node):
        replacements = []
        counters = {'VAR': 0, 'FUNC': 0, 'TYPE': 0}
        name_map = {}
        keywords = {'if', 'else', 'for', 'while', 'return', 'class', 'def', 'void', 'int', 'float', 'double', 'string', 'public', 'private', 'static', 'import', 'from', 'include', 'var', 'let', 'const', 'try', 'catch', 'true', 'false', 'null'}

        def traverse(node):
            is_id = ("identifier" in node.type or "variable" in node.type or "name" in node.type)
            if is_id and len(node.children) == 0:
                original_name = code_bytes[node.start_byte:node.end_byte].decode('utf8', errors='ignore')
                if original_name not in keywords and len(original_name) > 1:
                    parent_type = node.parent.type if node.parent else ""
                    category = 'VAR'
                    if ('function' in parent_type or 'method' in parent_type or 'call' in parent_type): category = 'FUNC'
                    elif ('class' in parent_type or 'type' in parent_type): category = 'TYPE'

                    if original_name not in name_map:
                        name_map[original_name] = f"{category}_{counters[category]}"
                        counters[category] += 1
                    replacements.append((node.start_byte, node.end_byte, name_map[original_name]))
            for child in node.children: traverse(child)

        traverse(root_node)
        replacements.sort(key=lambda x: x[0], reverse=True)
        mutable_code = bytearray(code_bytes)
        for start, end, new_text in replacements: mutable_code[start:end] = new_text.encode('utf8')
        return mutable_code

    def _flatten_layout(self, code_str, lang_type):
        lines = code_str.split('\n')
        processed_lines = []
        comment_pattern = re.compile(r'//.*|#.*|/\*.*?\*/', re.DOTALL)
        for line in lines:
            line = re.sub(comment_pattern, '', line)
            line = line.replace('\t', '    ').rstrip() if 'python' in lang_type else line.strip()
            if line: processed_lines.append(line)
        return "\n".join(processed_lines)

# --- 3. CUSTOM MODEL (MEAN-MAX POOLING) ---
class CustomUnixCoder(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size * 2, config.num_labels)
        self.init_weights()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        seq_out = outputs[0]
        mask = attention_mask.unsqueeze(-1).expand(seq_out.size()).float()
        mean_p = torch.sum(seq_out * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        seq_out_copy = seq_out.clone()
        seq_out_copy[attention_mask == 0] = -1e9
        max_p, _ = torch.max(seq_out_copy, 1)
        logits = self.classifier(self.dropout(torch.cat((mean_p, max_p), 1)))
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return (loss, logits) if loss is not None else logits

# --- 4. DATA PREP (Task C with Canonicalization) ---
print("[INFO] Loading & Canonicalizing Data...")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
canon = UniversalCanonicalizer()

def load_and_prep(path, is_test=False):
    df = pd.read_parquet(path)
    if not is_test: df = df.dropna(subset=['code', 'label'])
    # Uygulama: Canonicalization
    df['code'] = df.apply(lambda x: canon.canonicalize(x['code'], x.get('language', 'python')), axis=1)
    return df

train_df = load_and_prep(os.path.join(BASE_DIR, 'train.parquet'))
val_df = load_and_prep(os.path.join(BASE_DIR, 'validation.parquet'))
test_df = load_and_prep(os.path.join(BASE_DIR, 'test.parquet'), is_test=True)

def tokenize_fn(x): return tokenizer(x["code"], truncation=True, max_length=MAX_LENGTH)

ds_train = Dataset.from_pandas(train_df[['code', 'label']]).map(tokenize_fn, batched=True, num_proc=4)
ds_val = Dataset.from_pandas(val_df[['code', 'label']]).map(tokenize_fn, batched=True, num_proc=4)
ds_test = Dataset.from_pandas(test_df[['code']]).map(tokenize_fn, batched=True, num_proc=4)

for ds in [ds_train, ds_val, ds_test]:
    if "label" in ds.column_names: ds = ds.rename_column("label", "labels")
    ds.set_format("torch")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] Loading & Canonicalizing Data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/tree_sitter/__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


Map (num_proc=4):   0%|          | 0/900000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/200000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:

# --- 5. EXECUTION ---
model = CustomUnixCoder.from_pretrained(MODEL_NAME, num_labels=4)

args = TrainingArguments(
    output_dir="./mining_taskc", num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS, eval_strategy="steps", eval_steps=EVAL_STEPS,
    fp16=True, report_to="none"
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    _, _, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='macro', zero_division=0)
    return {'f1_macro': f1}

trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  compute_metrics=compute_metrics, data_collator=DataCollatorWithPadding(tokenizer))

trainer.train()

# 1. Hard Negative Mining
out_train = trainer.predict(ds_train)
losses = F.cross_entropy(torch.tensor(out_train.predictions), torch.tensor(out_train.label_ids), reduction='none')
hard_indices = torch.sort(losses, descending=True)[1][:int(len(ds_train) * 0.20)].tolist()

with open("/content/drive/MyDrive/Task C/hard_negative_indices.json", "w") as f:
    json.dump({"hard_indices": hard_indices}, f)

# 2. Submission CSV
out_test = trainer.predict(ds_test)
test_preds = np.argmax(out_test.predictions, axis=1)
pd.DataFrame({'id': test_df.get('id', test_df.index), 'label': test_preds}).to_csv(os.path.join(BASE_DIR, 'submission_mining_taskc.csv'), index=False)

print("✅ Part 1 Completed: JSON and CSV saved.")

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Some weights of CustomUnixCoder were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Step,Training Loss,Validation Loss,F1 Macro
1000,0.713300,0.584669,0.631840
2000,0.573300,0.539183,0.677180
3000,0.536800,0.517288,0.676182
4000,0.519800,0.502881,0.678005
5000,0.500600,0.488592,0.696732
6000,0.487100,0.488603,0.697020
7000,0.482500,0.484431,0.697761


✅ Part 1 Completed: JSON and CSV saved.
